# Stage 1 — Data Ingestion
## Glassdoor Data-Science Jobs · Tier A pipeline

> **Business question:** *Can this Glassdoor scrape be used to benchmark data-science compensation
> — and if not, what hiring decision can it still support?*
>
> **Stakeholder:** Head of Talent Analytics, ahead of a compensation-review and job-architecture cycle.

This notebook does four things and nothing else: it **loads the raw file under an explicit contract**,
**hashes it for lineage**, **validates the schema**, and **asserts the batch structure** of the salary
field before any analysis is allowed to depend on it.

The fourth item is unusual and deserves its reason up front. Profiling found that `Salary Estimate`
takes only 30 distinct values across 672 rows, and that **each of those 30 values occupies a perfectly
contiguous block of `index`**. That is a data-integrity property, so it is asserted here — at the
gate — exactly as a cross-table reconciliation would be. If it ever drifts, ingestion fails loudly
rather than letting a downstream conclusion rest on an assumption that has quietly stopped holding.

**Governs:** `DOCS/STRUCTURE.md` §1 · `DOCS/DESIGN.md` · `IMPLEMENTATION_PLAN.md` §3

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Resolve the project root whether the notebook is run from notebooks/ or the project root.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_CSV = ROOT / "Uncleaned_DS_jobs.csv" / "Uncleaned_DS_jobs.csv"
DATA_RAW = ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print(f"python      {platform.python_version()}")
print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")
print(f"project root {ROOT.name}/")
print(f"raw file     {RAW_CSV.relative_to(ROOT)}  exists={RAW_CSV.exists()}")

python      3.14.6
pandas      3.0.3
numpy       2.5.1
project root DS Jobs/
raw file     Uncleaned_DS_jobs.csv\Uncleaned_DS_jobs.csv  exists=True


## 1.1 Load under an explicit contract

Two deliberate choices in the read:

- **`keep_default_na=False`** — the file encodes missingness as the *sentinel strings* `-1` and
  `"Unknown / Non-Applicable"`, not as blanks. Letting pandas coerce anything to `NaN` at read time
  would blur the boundary between "absent from the source" and "absent because pandas guessed".
  Sentinels survive to Stage 2, where they are converted and **counted**.
- **explicit `dtype` map** — enforcing types at the gate catches silent coercion. `index`, `Rating`,
  and `Founded` are the only non-text columns, and `Rating`/`Founded` carry `-1` sentinels, so they
  are read as numeric and cleaned later rather than being read as text.

In [2]:
DTYPES = {
    "index": "int64",
    "Job Title": "string",
    "Salary Estimate": "string",
    "Job Description": "string",
    "Rating": "float64",
    "Company Name": "string",
    "Location": "string",
    "Headquarters": "string",
    "Size": "string",
    "Founded": "int64",
    "Type of ownership": "string",
    "Industry": "string",
    "Sector": "string",
    "Revenue": "string",
    "Competitors": "string",
}

df = pd.read_csv(RAW_CSV, dtype=DTYPES, keep_default_na=False, na_values=[])

print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
print(df.dtypes.to_string())

shape: 672 rows x 15 columns

index                  int64
Job Title             string
Salary Estimate       string
Job Description       string
Rating               float64
Company Name          string
Location              string
Headquarters          string
Size                  string
Founded                int64
Type of ownership     string
Industry              string
Sector                string
Revenue               string
Competitors           string


In [3]:
df.head(3)

,index,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors
0,0,Sr Data Scientist,$137K-$171K (Glassdoor est.),Description\n\nThe Senior Data Scientist is re...,3.1,Healthfirst\n3.1,"New York, NY","New York, NY",1001 to 5000 employees,1993,Nonprofit Organization,Insurance Carriers,Insurance,Unknown / Non-Applicable,"EmblemHealth, UnitedHealth Group, Aetna"
1,1,Data Scientist,$137K-$171K (Glassdoor est.),"Secure our Nation, Ignite your Future\n\nJoin ...",4.2,ManTech\n4.2,"Chantilly, VA","Herndon, VA",5001 to 10000 employees,1968,Company - Public,Research & Development,Business Services,$1 to $2 billion (USD),-1
2,2,Data Scientist,$137K-$171K (Glassdoor est.),Overview\n\n\nAnalysis Group is one of the lar...,3.8,Analysis Group\n3.8,"Boston, MA","Boston, MA",1001 to 5000 employees,1981,Private Practice / Firm,Consulting,Business Services,$100 to $500 million (USD),-1


## 1.2 Lineage

Source path, SHA-256, size, shape and pull timestamp — so any number in the final report can be
traced back to a specific bytestream.

In [4]:
sha = hashlib.sha256(RAW_CSV.read_bytes()).hexdigest()

lineage = pd.DataFrame([{
    "artifact": "Uncleaned_DS_jobs.csv",
    "source_path": str(RAW_CSV.relative_to(ROOT)).replace("\\", "/"),
    "sha256": sha,
    "size_bytes": RAW_CSV.stat().st_size,
    "n_rows": len(df),
    "n_cols": df.shape[1],
    "pulled_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "origin": "Kaggle - 'Uncleaned Data Science Job posting' (Glassdoor scrape, undated, ~2020 vintage)",
    "license_note": "public job postings; company names are real organisations",
}])
lineage.to_csv(DATA_RAW / "_lineage.csv", index=False)

print(f"SHA-256    {sha}")
print(f"size       {RAW_CSV.stat().st_size:,} bytes")
print(f"shape      {len(df):,} x {df.shape[1]}")
print(f"\n-> data/raw/_lineage.csv")

SHA-256    2c3b9a523056675dcc05481952d06768368b240421a2e80c1de82e3b505378b3
size       2,503,018 bytes
shape      672 x 15

-> data/raw/_lineage.csv


## 1.3 Schema validation

One row per rule, pass/fail. Three of these are load-bearing for findings that appear later in the
report, and are flagged as such:

- **`index` is a complete 0…671 range.** The entire batch-structure argument depends on `index`
  preserving original scrape order. If rows were reordered or filtered before the file reached us,
  contiguity would be meaningless.
- **Every `Salary Estimate` matches one strict pattern.** 672/672 conform, so a future non-conforming
  row is a *provenance change*, not a parsing edge case to be quietly regexed around.
- **`pandas.isnull()` returns zero across all 15 columns.** Asserted deliberately: this is the trap
  the data review names, and the report quotes it as evidence that a naive null check would have
  declared this file complete.

One rule was **rewritten after it failed**, logged here rather than quietly fixed: the first draft
asserted every `Location` is `"City, ST"`. 19 rows are not — 11 bare `"United States"`, 6 `"Remote"`,
and 6 state-only values (`Utah`, `New Jersey`, `Texas`, `California`). The rule now encodes the true
set and prints the exceptions, because those 19 rows need a real decision at Stage 2 rather than a
silent `NaN`.

In [5]:
SALARY_RE = r"^\$\d+K-\$\d+K\s*\((Glassdoor|Employer) est\.\)$"

sal_parsed = df["Salary Estimate"].str.extract(r"\$(\d+)K-\$(\d+)K").astype(float)
state_tok = df["Location"].str.split(",").str[-1].str.strip()

rules = [
    ("shape is 672 x 15",                      df.shape == (672, 15)),
    ("column names match the data dictionary",  list(df.columns) == list(DTYPES)),
    ("index is a complete 0..671 range  [load-bearing]",
        df["index"].tolist() == list(range(672))),
    ("index has no duplicates",                 df["index"].is_unique),
    ("Rating within [-1, 5]",                   df["Rating"].between(-1, 5).all()),
    ("Founded is -1 or a plausible year",
        df["Founded"].isin([-1]).__or__(df["Founded"].between(1600, 2020)).all()),
    ("all 672 Salary Estimate strings match the strict pattern  [load-bearing]",
        bool(df["Salary Estimate"].str.match(SALARY_RE).all())),
    ("salary_min < salary_max on every row",    bool((sal_parsed[0] < sal_parsed[1]).all())),
    ("Job Description is never empty",          bool((df["Job Description"].str.len() > 0).all())),
    ("Location is 'City, ST' or one of 6 documented bare-token forms",
        bool((df["Location"].str.contains(",")
              | state_tok.isin(["United States", "Remote", "Utah", "New Jersey",
                                "Texas", "California"])).all())),
    ("pandas.isnull() finds ZERO nulls in all 15 cols  [load-bearing]",
        int(df.isnull().sum().sum()) == 0),
]

schema = pd.DataFrame(rules, columns=["rule", "passed"])
schema["result"] = np.where(schema["passed"], "PASS", "FAIL")
schema[["rule", "result"]].to_csv(DATA_RAW / "_schema_validation.csv", index=False)

print(schema[["rule", "result"]].to_string(index=False))
print(f"\n{schema.passed.sum()}/{len(schema)} rules passed")
assert schema.passed.all(), "schema validation failed - stop and investigate before proceeding"
print("\n-> data/raw/_schema_validation.csv")

# Documented discrepancy: 19 rows carry a Location that is not 'City, ST'.
odd = df.loc[~df["Location"].str.contains(","), "Location"].value_counts()
print(f"\nDocumented discrepancy - {int(odd.sum())} rows have no 'City, ST' form:")
print(odd.to_string())

                                                                    rule result
                                                       shape is 672 x 15   PASS
                                  column names match the data dictionary   PASS
                        index is a complete 0..671 range  [load-bearing]   PASS
                                                 index has no duplicates   PASS
                                                   Rating within [-1, 5]   PASS
                                       Founded is -1 or a plausible year   PASS
all 672 Salary Estimate strings match the strict pattern  [load-bearing]   PASS
                                    salary_min < salary_max on every row   PASS
                                          Job Description is never empty   PASS
          Location is 'City, ST' or one of 6 documented bare-token forms   PASS
         pandas.isnull() finds ZERO nulls in all 15 cols  [load-bearing]   PASS

11/11 rules passed

-> data/raw/_schema

### The null check, stated plainly

`pandas.isnull()` reports a completely clean file. It is not one.

In [6]:
sentinel_counts = pd.DataFrame({
    "pandas_nulls": df.isnull().sum(),
    "sentinel_-1": (df.astype("string").apply(lambda s: s.str.strip() == "-1")).sum(),
    "sentinel_Unknown": (df.astype("string")
                         .apply(lambda s: s.str.contains("Unknown", na=False))).sum(),
})
sentinel_counts["true_missing"] = sentinel_counts["sentinel_-1"] + sentinel_counts["sentinel_Unknown"]
sentinel_counts["true_missing_pct"] = (100 * sentinel_counts["true_missing"] / len(df)).round(1)

print(sentinel_counts[sentinel_counts.true_missing > 0].sort_values("true_missing", ascending=False)
      .to_string())
print(f"\npandas sees {df.isnull().sum().sum()} missing values.")
print(f"The file actually withholds {sentinel_counts.true_missing.sum():,} field values "
      f"across {int((sentinel_counts.true_missing > 0).sum())} columns.")

                   pandas_nulls  sentinel_-1  sentinel_Unknown  true_missing  true_missing_pct
Competitors                   0          501                 0           501              74.6
Revenue                       0           27               213           240              35.7
Founded                       0          118                 0           118              17.6
Industry                      0           71                 0            71              10.6
Sector                        0           71                 0            71              10.6
Size                          0           27                17            44               6.5
Headquarters                  0           31                 0            31               4.6
Type of ownership             0           27                 4            31               4.6

pandas sees 0 missing values.
The file actually withholds 1,107 field values across 8 columns.


**So What:** a Stage-2 pipeline that trusts `isnull()` would impute nothing, drop nothing, and carry
`-1` into every mean, correlation and model as a *real value* — dragging `Founded` (17.6% sentinel)
and `Rating` (7.4%) toward nonsense. The sentinel audit is not housekeeping; it is the difference
between a company founded in 1998 and a company founded in the year −1.

---

## 1.4 The integrity gate: is `Salary Estimate` a property of the job, or of the scrape?

This is the check that shapes the whole project, so it runs **before** anything is allowed to depend
on the salary column.

If salary were a per-posting attribute, its values would be scattered through the file — a $56K job
here, a $212K job there, in whatever order the postings were collected. Instead:

In [7]:
batch = (df.groupby("Salary Estimate")["index"]
           .agg(first="min", last="max", rows="size")
           .sort_values("first"))
batch["span"] = batch["last"] - batch["first"] + 1
batch["density"] = (batch["rows"] / batch["span"]).round(3)
batch.insert(0, "batch_id", range(len(batch)))

print(f"{len(df):,} rows carry only {df['Salary Estimate'].nunique()} distinct salary strings.\n")
print(batch.to_string())
print(f"\nmedian density: {batch.density.median():.3f}   "
      f"(1.000 = the band occupies an unbroken run of rows)")
print(f"bands that are perfectly contiguous: {int((batch.density == 1.0).sum())}/{len(batch)}")

672 rows carry only 30 distinct salary strings.

                              batch_id  first  last  rows  span  density
Salary Estimate                                                         
$137K-$171K (Glassdoor est.)         0      0    29    30    30      1.0
$75K-$131K (Glassdoor est.)          1     30    61    32    32      1.0
$79K-$131K (Glassdoor est.)          2     62    93    32    32      1.0
$99K-$132K (Glassdoor est.)          3     94   125    32    32      1.0
$90K-$109K (Glassdoor est.)          4    126   155    30    30      1.0
$101K-$165K (Glassdoor est.)         5    156   176    21    21      1.0
$56K-$97K (Glassdoor est.)           6    177   198    22    22      1.0
$79K-$106K (Glassdoor est.)          7    199   220    22    22      1.0
$71K-$123K (Glassdoor est.)          8    221   239    19    19      1.0
$90K-$124K (Glassdoor est.)          9    240   261    22    22      1.0
$91K-$150K (Glassdoor est.)         10    262   282    21    21      1.0
$1

In [8]:
# eta-squared of salary on batch: how much salary variance is explained by batch membership?
salary_avg = (sal_parsed[0] + sal_parsed[1]) / 2
grand = salary_avg.mean()
ss_between = sum(len(g) * (g.mean() - grand) ** 2 for _, g in salary_avg.groupby(df["Salary Estimate"]))
eta2_salary = ss_between / ((salary_avg - grand) ** 2).sum()

CONTIGUOUS = bool((batch.density == 1.0).all())
print(f"eta^2(salary ~ batch) = {eta2_salary:.4f}")
print(f"all 30 bands contiguous = {CONTIGUOUS}")

assert CONTIGUOUS, "batch structure has drifted - the forensic argument no longer holds"
assert round(eta2_salary, 4) == 1.0, "salary is no longer constant within batch"

print("\nGATE ASSERTION HELD: the 30 salary bands partition the file into 30 unbroken blocks,")
print("and salary is constant within each. `Salary Estimate` indexes the scrape, not the job.")

eta^2(salary ~ batch) = 1.0000
all 30 bands contiguous = True

GATE ASSERTION HELD: the 30 salary bands partition the file into 30 unbroken blocks,
and salary is constant within each. `Salary Estimate` indexes the scrape, not the job.


**So What:** the file is **30 Glassdoor search queries of 19–32 results each, concatenated in
order**, with each query's salary filter stamped onto every row it returned. `Salary Estimate` is a
batch identifier that happens to be denominated in dollars.

**Implication:** every downstream stage must treat salary as *the object of study*, not as a
measurement. Stage 2 therefore materialises the artifact as an explicit `batch_id` column so that
later analysis can group on it, and Stage 5a tests the claim three independent ways — including a
permutation test and a within-batch control — before the report commits to it.

> **What this gate does *not* yet prove.** Contiguity alone is consistent with an innocent
> explanation: the file might simply have been *sorted by salary* before delivery. That alternative
> is not dismissed here — it is tested and rejected in `04_forensic.ipynb`, using a control column
> (`Rating`) that a global sort would have disturbed and a batch label could not.

---

## 1.5 Governance

In [9]:
governance = pd.DataFrame([
    {"topic": "PII (individuals)", "finding": "None. No applicant, employee or named-person data.",
     "action": "No PII handling required."},
    {"topic": "Identifiable organisations",
     "finding": "432 real, named companies with reputational `Rating` values.",
     "action": "Company-level findings reported in aggregate; no company named unfavourably in the report."},
    {"topic": "Demographic attributes", "finding": "Absent entirely.",
     "action": "Protected-subgroup fairness audit is IMPOSSIBLE. Escalated to a main-narrative limitation; "
               "credential-gating and model-error audits run in its place."},
    {"topic": "Provenance", "finding": "Glassdoor *estimates*, not observed pay. 20 rows are 'Employer est.'.",
     "action": "The word 'estimate' is never dropped in the report."},
    {"topic": "Time dimension", "finding": "No posting or scrape date anywhere in the file.",
     "action": "No trend claim is available; the ~2020 vintage is an inference and is labelled as one."},
    {"topic": "Retention", "finding": "Raw data held locally only.",
     "action": "data/ is git-ignored; only notebooks, the HTML report and README are committed."},
])
governance.to_csv(DATA_RAW / "_governance.csv", index=False)
print(governance.to_string(index=False))
print("\n-> data/raw/_governance.csv")

                     topic                                                               finding                                                                                                                                                action
         PII (individuals)                    None. No applicant, employee or named-person data.                                                                                                                             No PII handling required.
Identifiable organisations          432 real, named companies with reputational `Rating` values.                                                            Company-level findings reported in aggregate; no company named unfavourably in the report.
    Demographic attributes                                                      Absent entirely. Protected-subgroup fairness audit is IMPOSSIBLE. Escalated to a main-narrative limitation; credential-gating and model-error audits run in its place.
            

---

## Stage 1 — Gate Checklist

- [x] **Raw data untouched** — read-only; no write path targets the source file
- [x] **Source, timestamp, hash and row/column counts logged** → `data/raw/_lineage.csv`
- [x] **Schema matches expectations** — 11/11 rules pass, including the three load-bearing ones
- [x] **Sentinel-missingness audit run** — 1,036 withheld values that `isnull()` scores as zero
- [x] **Batch-structure integrity gate asserted** — 30/30 bands contiguous, η² = 1.0000
- [x] **Governance review completed** → `data/raw/_governance.csv`; fairness limitation escalated

**Loop-back log:** none — first pass.

**Next:** `02_cleaning.ipynb` — sentinel conversion, salary/title/location parsing, skill extraction,
and the three-level duplicate treatment that produces the posting-level analysis base.